# TUGAS PERTEMUAN 6 DATA SCIENCE
* **Nama Lengkap:** IKRAM
* **NIM:** 240401020139
* **Kelas:** IF 405


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

#memuat dataset Titanic resmi dari pustaka Seaborn
df = sns.load_dataset('titanic')

#memilih kolom-kolom yang wajib digunakan sesuai instruksi modul
cols = ['pclass','sex','age','sibsp','parch','fare','embarked','survived']
df = df[cols].copy()

print('Shape (Jumlah Baris & Kolom):', df.shape)
print('\nJumlah Missing Values (Data Kosong) per Kolom:')
print(df.isnull().sum())
print('\nProporsi Distribusi Target (Survived):')
print(df['survived'].value_counts(normalize=True).round(3))

Shape (Jumlah Baris & Kolom): (891, 8)

Jumlah Missing Values (Data Kosong) per Kolom:
pclass        0
sex           0
age         177
sibsp         0
parch         0
fare          0
embarked      2
survived      0
dtype: int64

Proporsi Distribusi Target (Survived):
survived
0    0.616
1    0.384
Name: proportion, dtype: float64


In [ ]:
#mengisi kolom 'age' yang kosong dengan nilai median
df['age'] = df['age'].fillna(df['age'].median())

#mengisi kolom 'embarked' yang kosong dengan nilai modus
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

print('Jumlah Missing Values setelah Handling:')
print(df.isnull().sum())

Jumlah Missing Values setelah Handling:
pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
survived    0
dtype: int64


In [ ]:
df = pd.get_dummies(df,
                    columns=['sex', 'embarked'],
                    drop_first=True,  #menghapus satu kolom kolom pertama untuk mencegah bias
                    dtype=int)        #menghasilkan angka 0 atau 1 (bukan True/False)

print('Kolom dataset setelah dilakukan Encoding:')
print(df.columns.tolist())

Kolom dataset setelah dilakukan Encoding:
['pclass', 'age', 'sibsp', 'parch', 'fare', 'survived', 'sex_male', 'embarked_Q', 'embarked_S']


In [ ]:
from sklearn.model_selection import train_test_split

#memisahkan Fitur (X) dan Target (y)
X = df.drop('survived', axis=1)
y = df['survived']

#membagi data menjadi 80% Train dan 20% Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  #agar proporsi kelas target seimbang
)

print(f'Jumlah data Train : {X_train.shape[0]} baris')
print(f'Jumlah data Test  : {X_test.shape[0]} baris')
print('\nProporsi target (Survived) di Data Train:')
print(y_train.value_counts(normalize=True).round(3))
print('\nProporsi target (Survived) di Data Test:')
print(y_test.value_counts(normalize=True).round(3))

Jumlah data Train : 712 baris
Jumlah data Test  : 179 baris

Proporsi target (Survived) di Data Train:
survived
0    0.617
1    0.383
Name: proportion, dtype: float64

Proporsi target (Survived) di Data Test:
survived
0    0.615
1    0.385
Name: proportion, dtype: float64


In [ ]:
from sklearn.preprocessing import StandardScaler

#menentukan kolom numerik yang wajib disamakan skalanya
num_cols = ['pclass', 'age', 'sibsp', 'parch', 'fare']
scaler = StandardScaler()

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

X_test[num_cols] = scaler.transform(X_test[num_cols])

print('Mean scaler (rata-rata dari data Train):', scaler.mean_.round(2))
print('Std scaler (standar deviasi dari data Train):', scaler.scale_.round(2))
print()
print('Contoh data X_train setelah skalanya disamakan:')
print(X_train.head(3).round(3))


Mean scaler (rata-rata dari data Train): [-0.  0. -0. -0. -0.]
Std scaler (standar deviasi dari data Train): [1. 1. 1. 1. 1.]

Contoh data X_train setelah skalanya disamakan:
     pclass    age  sibsp  parch   fare  sex_male  embarked_Q  embarked_S
692   0.830 -0.112 -0.465 -0.466  0.514         1           0           1
481  -0.371 -0.112 -0.465 -0.466 -0.663         1           0           1
527  -1.571 -0.112 -0.465 -0.466  3.955         1           0           1


### Kesimpulan Pertemuan 6 (Persiapan Data & Preprocessing)

* **Apa yang dipelajari:**
  Tahapan penting *Data Preprocessing* sebelum melatih model *Machine Learning*, meliputi pengisian data kosong (*imputation*), pengubahan data teks menjadi biner (*One-Hot Encoding*), pembagian data uji (*Train-Test Split*), dan standardisasi skala angka (*Feature Scaling*).

* **Temuan utama:**
  * Pengisian *missing value* pada fitur `age` dengan median dan `embarked` dengan modus berhasil membersihkan data agar siap diproses algoritma scikit-learn.
  * Penerapan parameter `drop_first=True` pada *One-Hot Encoding* sangat penting untuk menghindari bias multikolinieritas (*Dummy Variable Trap*).
  * Penggunaan parameter `stratify=y` saat membagi data (proporsi 80:20) berhasil menjaga keseimbangan porsi kelas target selamat/tidak selamat agar tetap seragam antara data latih dan data uji.

* **Keterbatasan / Tantangan yang muncul:**
  Proses *Feature Scaling* (`StandardScaler`) harus dipisahkan secara ketat, di mana parameter rata-rata dan deviasi standar hanya dipelajari dari data latih (`fit_transform`) lalu diterapkan ke data uji (`transform`). Tantangan utamanya adalah disiplin dalam alur ini agar terhindar dari kebocoran informasi (*data leakage*) yang bisa membuat performa model terlihat terlalu bagus secara semu (*overoptimistic*).
